# Data Exploration (B)


### Volume-Spike Target and Modelling Limits

We define a volume spike as a day whose volume is at least 1.5× the average of the previous 20 trading days for that ticker. With the provisional eight-ticker list, this produces 42 positive observations out of 512 rows, compared with 26 at 1.75× and 12 at 2×, supporting the concern in our Phase 1 proposal that the 2× definition would produce too few positive cases.

At 1.5×, TSLA (12.5%), PLTR (9.4%), and AAPL (9.4%) have higher spike rates than GME (7.8%) and BB (3.1%), which may partly reflect the January frenzy raising the meme stocks’ baseline volume. For example, in our data on January 29, GME traded 202 million shares against a 245 million baseline which is below its recent average during the squeeze, but aLSO showing how the chosen window can make unusually high activity look normal.

Using roughly ten positive cases per predictor as a planning guide suggests about four features before splitting the data and exploring volume momentum, mention velocity, sentiment, and combined emotion intensity, but we will revisit this once our coverage counts settle the ticker list and we know how many positive cases remain in the training sample.

In [4]:
# If this errors with "No module named yfinance", run this once in a cell first:

import pandas as pd
import yfinance as yf

gme = yf.download(
    "GME",              # the ticker, as a string
    start="2020-12-21",        # a date string "YYYY-MM-DD", about 20 trading days before Jan 29, 2021
    end="2021-05-01",          # careful: yfinance does NOT include the end date itself
    progress=False,
)

print(gme.shape)
gme.head()

(90, 5)


Price,Close,High,Low,Open,Volume
Ticker,GME,GME,GME,GME,GME
Date,,,,,
2020-12-21,3.8825,4.0875,3.8200,3.9525,39504400
2020-12-22,4.8650,5.0100,4.0375,4.0550,122610800
2020-12-23,5.1425,5.5875,4.7825,5.0425,103321200
2020-12-24,5.0375,5.3700,4.9875,5.2525,25048800
2020-12-28,5.2475,5.4925,5.0875,5.3275,35863600


In [5]:
# Calculate the 20-day rolling average of GME's daily trading volume, shifted down by one day to exclude the current day's volume from the average.

volume = gme["Volume"]["GME"]

# Normal level = average of the 20 trading days BEFORE each day.
# rolling(20).mean() includes today, so shift(1) moves every value down a row:
# each day now gets the average that ended yesterday.
avg20 = volume.rolling(20).mean().shift(1)

check = pd.DataFrame({"volume": volume, "avg20": avg20})
check.loc["2021-01-25":"2021-02-05"]

,volume,avg20
Date,,
2021-01-25,711496000,153086220.0
2021-01-26,714352000,183494960.0
2021-01-27,373586800,217960120.0
2021-01-28,235263200,234846280.0
2021-01-29,202264400,244761160.0
2021-02-01,149528800,253687500.0
2021-02-02,312732400,259779400.0
2021-02-03,170794000,273411520.0
2021-02-04,249709200,280958920.0


In [6]:
# The 20 trading days before Jan 29, averaged directly
by_hand = volume.loc["2020-12-30":"2021-01-28"]
print(len(by_hand), by_hand.mean())
print(check.loc["2021-01-29", "avg20"])

20 244761160.0
244761160.0


In [7]:
# How many times bigger than normal was each day?
check["ratio"] = check["volume"] / check["avg20"]

# Spike flags at the three thresholds (True/False)
for t in [2.0, 1.75, 1.5]:
    check[f"spike_{t}"] = check["ratio"] >= t

# Keep only the analysis window
window = check.loc["2021-01-29":"2021-04-30"]
print(len(window))   # should be 64

# Share of days that spiked at each threshold
window[["spike_2.0", "spike_1.75", "spike_1.5"]].mean().round(3)

64


spike_2.0     0.031
spike_1.75    0.062
spike_1.5     0.078
dtype: float64

In [8]:
# Run the same analysis for all the candidate tickers, then combine the results into one DataFrame.

TICKERS = ["GME", "AMC", "BB", "NOK", "TSLA", "PLTR", "AMD", "AAPL"]

def build_target(volume, window=20):
    """Volume ratio vs. the prior `window` trading days, plus spike flags."""
    out = pd.DataFrame({"volume": volume})
    out["avg20"] = volume.rolling(window).mean().shift(1)
    out["ratio"] = out["volume"] / out["avg20"]
    for t in [2.0, 1.75, 1.5]:
        out[f"spike_{t}"] = out["ratio"] >= t
    return out

prices = yf.download(TICKERS, start="2020-12-21", end="2021-05-01", progress=False)

frames = []
for tk in TICKERS:
    t = build_target(prices["Volume"][tk])
    t["ticker"] = tk
    frames.append(t.loc["2021-01-29":"2021-04-30"])

target = pd.concat(frames)
print(target.shape)   # should be (512, 7): 64 days x 8 tickers

spike_cols = ["spike_2.0", "spike_1.75", "spike_1.5"]
by_ticker = target.groupby("ticker")[spike_cols].mean().round(3)
by_ticker.loc["POOLED"] = target[spike_cols].mean().round(3)
by_ticker

(512, 7)


,spike_2.0,spike_1.75,spike_1.5
ticker,,,
AAPL,0.000,0.016,0.094
AMC,0.016,0.094,0.109
AMD,0.016,0.016,0.062
BB,0.000,0.031,0.031
GME,0.031,0.062,0.078
NOK,0.031,0.047,0.062
PLTR,0.047,0.078,0.094
TSLA,0.047,0.062,0.125
POOLED,0.023,0.051,0.082
